In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display


%load_ext autoreload
%autoreload 2

from vpop_calibration import *

In [ ]:
df = pd.read_csv("Mavoglurant_Benchmark_Dataset.csv")


obs_df = (
    df.loc[df["EVID"] == 0]
      .rename(
          columns={
              "ID": "id",
              "TIME": "time",
              "DV": "value",
              "DOSE": "dose"

          }
      )
      [["id", "time", "value", "dose", "WT"]]
      .astype({"value": "float"})
)
obs_df["time"] = obs_df["time"].apply(lambda t: t * 60 * 60)
obs_df["value"]= obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "logC15"
obs_df["protocol_arm"] = "identity"
display(df.head())
display(obs_df.head())

In [ ]:
model = SimworkModelBinding(
    path_to_model="cm.json",
    path_to_solving_options="sv.json",
    inputs=["KbBR", "CLint","KbMU","KbAD","KbBO","KbRB","dose","WT"],
    outputs=["logC15"],
)
print(model.inputs)

struct_model = StructuralSimwork(model=model)

In [ ]:
prior_pdu = {
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 0.5},
        "KbBR": {"prior": np.exp(0.5),"prior_omega": 0.25},
        "KbMU": {"prior": np.exp(0.3),"prior_omega": 0.25}, 
        "KbBO": {"prior": np.exp(0.03),"prior_omega": 0.25},
        "KbAD":  {"prior": np.exp(2), "prior_omega": 0.25},
        "KbRB": {"prior": np.exp(0.3), "prior_omega": 0.25},
    },
    "pdk": {
        "WT",
        "dose"
    },
    "error_model": {
        "logC15": {"error_type": "additive", "sigma": 0.5},
    },
}

prior_MI = {
    "model_intrinsic": {
        "KbBR": {"prior": np.exp(0.5)},
        "KbMU": {"prior": np.exp(0.3)},
        "KbBO": {"prior": np.exp(0.03)},
        "KbAD":  {"prior": np.exp(2)},
        "KbRB": {"prior": np.exp(0.3)},
    },
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 0.5},
    },
    "pdk": {
        "WT",
        "dose"
    },
    "error_model": {
        "logC15": {"error_type": "additive", "sigma": 0.5},
    },
}

In [ ]:
def fit_saem(prior)-> NlmeModel:
    config = Config(
        saem=SaemConfigDict(
            nb_phase1_iterations=100,
            nb_phase2_iterations=100,
            plot_frames=5,
            optim_max_fun=2,
            verbose=False,
            mcmc_first_burn_in=0,
        ),
        nlme=NlmeConfigDict(nb_chains=1),
    )
    nlme_model = NlmeModel(df=obs_df, prior_params=prior, structural_model=struct_model, config=config)
    nlme_model.optimizer.run()
    return nlme_model

In [ ]:
%%capture output
%timeit -n 1 -r 5 -v time_saem_MI fit_saem(prior_MI)

In [ ]:
perf_df = pd.DataFrame({
    "time_saem": time_saem_MI.timings,
})

perf_df.to_csv(
    "Mavoglurant_Simwork_MI_runtime.csv",
    index=False
)

perf_df

In [ ]:
%%capture output
%timeit -n 1 -r 5 -v time_saem_PDU fit_saem(prior_pdu)

In [ ]:
perf_df = pd.DataFrame({
    "time_saem": time_saem_PDU.timings,
})

perf_df.to_csv(
    "Mavoglurant_Simwork_PDU_runtime.csv",
    index=False
)

perf_df